# Week 4 Exercise — Python to JavaScript Converter

This notebook implements a simple AI-powered code converter using an LLM. 
Users can paste Python code into the interface, and the application will 
generate equivalent JavaScript code. The system uses OpenRouter to access 
a language model and Gradio to provide a lightweight user interface.

The goal of this exercise is to demonstrate how to integrate an LLM into 
a small application for code generation using prompt engineering and a 
basic interactive UI.



In [ ]:
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)


In [ ]:
SYSTEM_PROMPT = """
You are an expert software engineer.

Convert Python code into equivalent JavaScript code.

Requirements:
- Preserve the same program structure
- Preserve functions and variables
- Preserve logic and flow
- Do not simplify the program
- Do not remove functions
- Do not inline values
- Output ONLY JavaScript code
- No markdown or explanations
- Use modern JavaScript (ES6+)
"""

MODEL = "meta-llama/llama-3.1-8b-instruct"


def python_to_js(python_code):

    if not python_code.strip():
        return "// Paste Python code above"

    prompt = f"""
Convert this Python code to JavaScript.

Python code:
{python_code}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    js_code = response.choices[0].message.content.strip()

    # remove accidental markdown if model adds it
    js_code = js_code.replace("```javascript", "").replace("```", "")

    return js_code

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("## Python → JavaScript Converter")

    with gr.Row():
        python_input = gr.Code(
            label="Python Code",
            language="python",
            value="""def greet(name):
    print(f"Hello {name}")

greet("World")
"""
        )

        js_output = gr.Code(
            label="JavaScript Output",
            language="javascript"
        )

    convert_btn = gr.Button("Convert")

    convert_btn.click(
        python_to_js,
        inputs=python_input,
        outputs=js_output
    )

demo.launch()